# Incident response — before / after / after-disrupted

Notebook version of the three-script screencast in [`examples/screencast/incident/`](../../examples/screencast/incident/). Three runs of the same scenario: a hand-wired LangGraph baseline, the GOAP refactor on the happy path, and the GOAP refactor under cascading disruption. Backed by [`tests/integration/test_screencast_incident.py`](../../../tests/integration/test_screencast_incident.py).

## 1. Action specs

Six recovery actions plus a `notify_stakeholders` step. Cost numerics encode escalation priority: restart(1) → rollback(2) → analyze(1)+hotfix(3) → failover(4). All six share the same precondition (`incident_detected=True`).

In [1]:
from typing import Any
from langgoap.actions import ActionSpec

def restart_service(ws):
    if ws.get("restart_ineffective"):
        raise RuntimeError("OOM from memory leak")
    return {"service_healthy": True, "recovery_method": "restart"}

def rollback_deployment(ws):
    if ws.get("rollback_blocked"):
        raise RuntimeError("irreversible DB migration")
    return {"service_healthy": True, "recovery_method": "rollback"}

def analyze_error_logs(ws):
    return {"root_cause_hypothesized": True,
            "root_cause": "memory leak in RequestHandler cache"}

def apply_hotfix(ws):
    return {"service_healthy": True, "recovery_method": "hotfix"}

def failover_to_backup(ws):
    return {"service_healthy": True, "recovery_method": "failover"}

def notify_stakeholders(ws):
    return {"stakeholders_notified": True}

incident_actions = [
    ActionSpec(name="restart_service", cost=1.0,
               preconditions={"incident_detected": True},
               effects={"service_healthy": True},
               execute=restart_service),
    ActionSpec(name="rollback_deployment", cost=2.0,
               preconditions={"incident_detected": True},
               effects={"service_healthy": True},
               execute=rollback_deployment),
    ActionSpec(name="analyze_error_logs", cost=1.0,
               preconditions={"incident_detected": True},
               effects={"root_cause_hypothesized": True},
               execute=analyze_error_logs),
    ActionSpec(name="apply_hotfix", cost=3.0,
               preconditions={"root_cause_hypothesized": True},
               effects={"service_healthy": True},
               execute=apply_hotfix),
    ActionSpec(name="failover_to_backup", cost=6.0,
               preconditions={"incident_detected": True},
               effects={"service_healthy": True},
               execute=failover_to_backup),
    ActionSpec(name="notify_stakeholders", cost=1.0,
               preconditions={"service_healthy": True},
               effects={"stakeholders_notified": True},
               execute=notify_stakeholders),
]
[a.name for a in incident_actions]

/Users/brian.sam-bodden/Code/langgoap/.venv/lib/python3.12/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


['restart_service',
 'rollback_deployment',
 'analyze_error_logs',
 'apply_hotfix',
 'failover_to_backup',
 'notify_stakeholders']

## 2. Happy path — planner picks the cheapest chain

`restart_service` (cost 1) satisfies `service_healthy=True` and is the cheapest single-action path; the planner pairs it with `notify_stakeholders`.

In [2]:
from langgoap.goals import GoalSpec
from langgoap.graph.builder import GoapGraph

goal = GoalSpec(conditions={
    "service_healthy": True,
    "stakeholders_notified": True,
})
graph = GoapGraph(incident_actions)

result = graph.invoke(
    goal=goal,
    world_state={"incident_detected": True},
)
[(h.action_name, h.success) for h in result["execution_history"]]

[('restart_service', True), ('notify_stakeholders', True)]

## 3. Disruption — restart is ineffective

In [3]:
result = graph.invoke(
    goal=goal,
    world_state={"incident_detected": True, "restart_ineffective": True},
)
[(h.action_name, h.success) for h in result["execution_history"]]

Action 'restart_service' failed: OOM from memory leak


[('restart_service', False),
 ('rollback_deployment', True),
 ('notify_stakeholders', True)]

## 4. Cascading disruption — restart + rollback both fail

In [4]:
result = graph.invoke(
    goal=goal,
    world_state={
        "incident_detected": True,
        "restart_ineffective": True,
        "rollback_blocked": True,
    },
)
[(h.action_name, h.success) for h in result["execution_history"]]

Action 'restart_service' failed: OOM from memory leak


Action 'rollback_deployment' failed: irreversible DB migration


[('restart_service', False),
 ('rollback_deployment', False),
 ('analyze_error_logs', True),
 ('apply_hotfix', True),
 ('notify_stakeholders', True)]

Three runs, three different paths, **one set of action definitions**. The planner discovers each path at runtime from the same declarative specs. Compare with the hand-wired baseline in [`before.py`](../../examples/screencast/incident/before.py).